In [43]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 

import seaborn as sns
import os

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer



sns.set_style('whitegrid')

%matplotlib inline

In [ ]:
df = pd.read_csv('../data/raw/bank_churn_raw.csv')

# display(df.head(20))
print(df.shape)
display(df.info())
display(df.describe())
display(df.dtypes)
display(df.sample(20, random_state=42))
display(df.isnull().sum().sort_values(ascending=False))

In [ ]:
drop_cols = ['customer_id']

object_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
print(object_cols)

for col in object_cols:
    n_unique = df[col].nunique()
    print(f"{col}: {n_unique} unique values")
    if n_unique < 20:
        print(f"Value counts for {col}:\n{df[col].value_counts()}\n")
        
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate customer_id:", df['customer_id'].duplicated().sum())

In [ ]:
red_flags = {}

leak_candidates = ['exit_survey_score', 'account_closed_date', 'churn_flag_internal']

for c in leak_candidates:
    if c in df.columns:
        red_flags[c] = {
            "null_rate": df[c].isna().mean(),
            "non_null_if_churn": df.loc[df['churned'] == 1, c].notna().mean(),
            "non_null_if_not_churn": df.loc[df['churned'] == 0, c].notna().mean()            
        }
income_sample = df['annual_income'].astype(str).sample(30, random_state=42)
# display(income_sample.describe())

target_dist = df['churned'].value_counts(dropna=False)
target_ratio = df['churned'].value_counts(normalize=True, dropna=False)


print("Leakage check results:", red_flags)
print("Target distribution:", target_dist)
print("Target ratio:", target_ratio)
print("Annual Income sample:\n")
for x in income_sample:
    print(x)


In [ ]:
target_count = df['churned'].value_counts()
target_ratio = df['churned'].value_counts(normalize=True)

neg = target_count.get(0,0)
pos = target_count.get(1,0)

imbalance_ratio = neg / max(pos, 1)  # Avoid division by zero


print("Target distribution:\n", target_count)
print("Target ratio:\n", target_ratio)
print(f"Imbalance ratio (0 : 1): {imbalance_ratio:.2f} : 1")

In [ ]:
rows = []

for col in df.columns:
    if df[col].isna().sum() == 0:
        continue
    
    null_if_churn = df.loc[df['churned'] == 1, col].isna().mean()
    null_if_not = df.loc[df['churned'] == 0, col].isna().mean()
    abs_diff = abs(null_if_churn - null_if_not)
    
    
    rows.append({
        'column': col,
        'null_rate': df[col].isna().mean(),
        'null_if_churn': null_if_churn,
        'null_if_not_churn': null_if_not,
        'abs_diff': abs_diff
    })
    
miss_df = pd.DataFrame(rows).sort_values('abs_diff', ascending=False)
display(miss_df)

In [ ]:
leak_cols = ['exit_survey_score', 'account_closed_date', 'churn_flag_internal']
df_noleak = df.drop(columns=leak_cols, errors='ignore')

print("Original shape:", df.shape)
print("After leakage drop:", df_noleak.shape)
print("Dropped:", leak_cols)

In [47]:
df_copy_clean = df_noleak.copy()

# Identify columns with missing values (excluding target)
missing_cols = [c for c in df_copy_clean.columns if c != 'churned' and df_copy_clean[c].isna().sum() > 0]

results = []

for col in missing_cols:
    temp = df_copy_clean.copy()
    temp['missing_flag'] = temp[col].isna().astype(int)
    
    feat_cols = [c for c in temp.columns if c not in [col, 'missing_flag']]
    X = temp[feat_cols]
    y = temp['missing_flag']
    
    # Only evaluate if both classes are present
    if y.nunique() < 2:
        continue
    
    
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
    
    
    pre = ColumnTransformer(
        transformers = [
            ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
            ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
        ]
    )
    
    
    model = Pipeline([
        ('pre', pre),
        ('clf', LogisticRegression(max_iter=2000, random_state=42))
    ])
    
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    auc = cross_val_score(model, X, y, cv=cv, scoring='roc_auc').mean()
    
    
    gaps = abs(temp.loc[temp['churned'] == 1, col].isna().mean() - temp.loc[temp['churned'] == 0, col].isna().mean())
    
    if auc < 0.55 and gaps < 0.02:
        label ='likely MCAR'
    elif auc >= 0.55 and gaps >= 0.02:
        label = 'likely MAR'
    elif auc >= 0.55 and gaps >= 0.02:
        label = 'Likely MNAR/MAR-strong'
    else:
        label = 'unclear'
        
        
    results.append({
        'column': col,
        'missing_rate': temp[col].isna().mean(),
        'missingness_auc_from_observed': auc,
        'gap_vs_target': gaps,
        'heuristic_label': label
    })
    
    
diag = pd.DataFrame(results).sort_values(['heuristic_label', 'missing_rate', 'missingness_auc_from_observed'], ascending=[True, False, False])
display(diag)

,column,missing_rate,missingness_auc_from_observed,gap_vs_target,heuristic_label
6,complaints_12mo,0.036019,0.582586,0.212942,likely MAR
8,digital_engagement,0.087379,0.516641,0.014655,likely MCAR
7,debt_to_income,0.086893,0.525190,0.015149,likely MCAR
9,has_credit_card,0.060680,0.532174,0.006899,likely MCAR
5,monthly_transactions,0.049320,0.530466,0.004647,likely MCAR
1,region,0.040583,0.547487,0.013529,likely MCAR
4,num_products,0.034660,0.487764,0.013462,likely MCAR
2,account_type,0.028058,0.518431,0.010261,likely MCAR
3,credit_score,0.081262,0.518569,0.039133,unclear
0,gender,0.071748,0.540960,0.036410,unclear
